**Note (post-reorganization):** this notebook reads from the `experiments/` PSBD-sweep
output format via `experiment_io`/`plotting`, both of which now live in `_archive/`
since `sweep.py`/`run_sweep.py` (the code that produced that output) were archived too.
This notebook needs updating once the PSBD sweep is rewritten.

## Imports

In [13]:
import glob
import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from lightning import seed_everything
from sklearn.model_selection import train_test_split

import torchvision.transforms.v2 as T
from torchvision import datasets
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import roc_auc_score, roc_curve

In [14]:
def clean_gpu_cache():
    import gc

    gc.collect()
    torch.cuda.empty_cache()


clean_gpu_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.bfloat16)
# torch.set_default_dtype(torch.float32)
print("Device:", device)

Device: cuda


## Config

In [15]:
SEED = 0
TRIGGER_LABEL = 0
BATCH_SIZE = 16
CLEAN_VAL_SIZE = 2000  # from train set, per PSBD paper (5% of training data)
EXAMPLES_PER_CLASS = 150  # balanced eval set per class
FORWARD_PASSES = 3  # k from paper Section 4.2
PSBD_QUANTILES = [0.1, 0.15, 0.2, 0.25]
DROPOUT_RATES = [i / 10.0 for i in range(1, 10)]
VIT_WEIGHTS_DIR = "vit_b_16_weights"
RAW_DATA_DIR = "raw_data"
# RESULTS_DIR = "/content/drive/MyDrive/PSBD/experiments"
DROPOUT_TYPE = "post_residual"  # "existing" or "post_residual"
RESULTS_DIR = "experiments/" + DROPOUT_TYPE
ATTACK_FOLDERS = [
    "cifar10_badnet_0_05",
    "cifar10_badnet_0_1",
    "cifar10_wanet_0_05",
    "cifar10_wanet_0_1",
    "cifar10_blended_0_05",
    "cifar10_blended_0_1",
    "cifar10_sig_0_05",
    "cifar10_sig_0_1",
    "cifar100_badnet_0_05",
    "cifar100_badnet_0_1",
    "cifar100_wanet_0_05",
    "cifar100_wanet_0_1",
    "cifar100_blended_0_05",
    "cifar100_blended_0_1",
    "cifar100_sig_0_05",
    "cifar100_sig_0_1",
    "gtsrb_badnet_0_05",
    "gtsrb_badnet_0_1",
    "gtsrb_wanet_0_05",
    "gtsrb_wanet_0_1",
    "gtsrb_blended_0_05",
    "gtsrb_blended_0_1",
]

## Dataset utilities

In [16]:
DATASET_CONFIG = {
    "cifar10": {
        "num_classes": 10,
        "mean": (0.4914, 0.4822, 0.4465),
        "std": (0.2023, 0.1994, 0.2010),
        "cls": datasets.CIFAR10,
    },
    "cifar100": {
        "num_classes": 100,
        "mean": (0.5071, 0.4867, 0.4408),
        "std": (0.2673, 0.2564, 0.2762),
        "cls": datasets.CIFAR100,
    },
    "gtsrb": {
        "num_classes": 43,
        "mean": (0.0, 0.0, 0.0),
        "std": (1.0, 1.0, 1.0),
        "cls": datasets.GTSRB,
    },
    "tiny": {
        "num_classes": 200,
        "mean": (0.4802, 0.4481, 0.3975),
        "std": (0.2302, 0.2265, 0.2262),
        # "mean": (0.485, 0.456, 0.406),
        # "std": (0.229, 0.224, 0.225),
        "cls": None,  # koristi ImageFolder
    },
}


def get_transform(dataset_name: str) -> T.Compose:
    cfg = DATASET_CONFIG[dataset_name]
    return T.Compose(
        [
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(mean=cfg["mean"], std=cfg["std"]),
        ]
    )


def denormalize(tensor, dataset_name: str):
    mean = DATASET_CONFIG[dataset_name]["mean"]
    std = DATASET_CONFIG[dataset_name]["std"]
    mean = torch.tensor(mean).view(-1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1)
    return tensor * std + mean


def load_clean_datasets(dataset_name: str, transform: T.Compose) -> tuple:
    cfg = DATASET_CONFIG[dataset_name]
    root = os.path.join(RAW_DATA_DIR, dataset_name)

    if dataset_name == "gtsrb":
        train_ds = datasets.GTSRB(
            root=root, split="train", download=True, transform=transform
        )
        test_ds = datasets.GTSRB(
            root=root, split="test", download=True, transform=transform
        )
    elif dataset_name == "tiny":
        # BackdoorBench sprema TinyImageNet kao standardnu ImageFolder strukturu:
        # raw_data/tinyimagenet/train/<class_id>/...
        # raw_data/tinyimagenet/val/<class_id>/...    ← BackdoorBench reorganizira val u klase
        train_ds = datasets.ImageFolder(
            os.path.join(root, "train"), transform=transform
        )
        test_ds = datasets.ImageFolder(os.path.join(root, "val"), transform=transform)

    else:
        train_ds = cfg["cls"](root=root, train=True, download=True, transform=transform)
        test_ds = cfg["cls"](root=root, train=False, download=True, transform=transform)

    return train_ds, test_ds


class PngPathDataset(torch.utils.data.Dataset):
    """Loads backdoor images from PNG files on disk (as saved by BackdoorBench)."""

    def __init__(self, paths: list, transform: T.Compose, label: int):
        self.paths = paths
        self.transform = transform
        self.label = label

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> tuple:
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.label


def get_backdoor_splits(
    folder_name: str,
    test_dataset: torch.utils.data.Dataset,
    transform: T.Compose,
    trigger_label: int = TRIGGER_LABEL,
) -> tuple:
    """
    Returns (bd_test, cl_test) where:
        bd_test          : backdoor test samples (PNG files with trigger)
        cl_test  : same original images WITHOUT trigger (for paired comparison)

    The filenames in bd_test_dataset/ are the original dataset indices,
    so we can recover the exact clean counterparts.
    """
    bd_test_dir = os.path.join(VIT_WEIGHTS_DIR, folder_name, "bd_test_dataset")
    bd_paths = sorted(glob.glob(f"{bd_test_dir}/**/*.png", recursive=True))

    if not bd_paths:
        raise FileNotFoundError(f"No PNG files found in {bd_test_dir}")

    # Extract original dataset indices from filenames (e.g. '49685.png' -> 49685)
    bd_indices = [int(Path(p).stem) for p in bd_paths]

    bd_test = PngPathDataset(bd_paths, transform=transform, label=trigger_label)
    cl_test = Subset(test_dataset, bd_indices)

    return bd_test, cl_test


def get_balanced_subset(
    cl_ds: torch.utils.data.Dataset,
    bd_ds: torch.utils.data.Dataset,
    examples_per_class: int,
    seed: int = SEED,
) -> tuple[Subset, Subset]:
    """Balanced subset with fixed number of examples per class."""
    if hasattr(cl_ds, "targets"):
        cl_labels = cl_ds.targets
    elif hasattr(cl_ds, "samples"):
        cl_labels = [y for _, y in cl_ds.samples]
    else:
        cl_labels = [cl_ds[i][1] for i in range(len(cl_ds))]

    class_to_indices: dict = {}
    for idx, y in enumerate(cl_labels):
        class_to_indices.setdefault(int(y), []).append(idx)

    seed_everything(seed)
    rng = np.random.default_rng(seed)
    selected = []
    for class_id in sorted(class_to_indices):
        candidates = class_to_indices[class_id]
        rng.shuffle(candidates)
        selected.extend(candidates[:examples_per_class])

    balanced_cl_ds = Subset(cl_ds, selected)
    balanced_bd_ds = Subset(bd_ds, selected)
    return balanced_cl_ds, balanced_bd_ds

## Model loading

In [ ]:
def load(checkpoint_path: str) -> nn.Module:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    num_classes = checkpoint["num_classes"]
    net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
    net.heads.head = nn.Linear(net.heads.head.in_features, num_classes)
    model = nn.Sequential(T.Resize((224, 224)), net)

    state_dict = checkpoint.get("model", checkpoint)
    if isinstance(state_dict, nn.Module):
        state_dict = state_dict.state_dict()
    # Strip 'module.' prefix from DataParallel checkpoints
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

    result = model.load_state_dict(state_dict, strict=False)
    if result.missing_keys:
        print(f"Missing keys: {len(result.missing_keys)}")
    if result.unexpected_keys:
        print(f"Unexpected keys: {len(result.unexpected_keys)}")

    model.to(device).eval()
    return model


@torch.inference_mode()
def compute_asr(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    dtype = next(model.parameters()).dtype
    device = next(model.parameters()).device
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        imgs, labels = imgs.to(dtype), labels.to(torch.long)
        correct += (model(imgs).argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0


def get_attack_file_paths(dir_path):
    train_img_paths = glob.glob(f"{dir_path}/bd_train_dataset/**/*.png", recursive=True)
    test_img_paths = glob.glob(f"{dir_path}/bd_test_dataset/**/*.png", recursive=True)
    print("Number of training images:", len(train_img_paths))
    print("Number of test images:", len(test_img_paths))
    return train_img_paths, test_img_paths


def configure_existing_dropouts(model: nn.Module, p: float) -> int:
    count = 0
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.p = float(p)
            m.train() if p > 0.0 else m.eval()
            count += 1
    return count


class PostResidualEncoderBlock(nn.Module):
    def __init__(self, base_block: nn.Module, p: float):
        super().__init__()
        self.base_block = base_block
        self.attn_dropout = nn.Dropout(p=p)  # after first residual add
        self.mlp_dropout = nn.Dropout(p=p)  # after second residual add

    def forward(self, input_tensor):
        x = self.base_block.ln_1(input_tensor)
        x, _ = self.base_block.self_attention(x, x, x, need_weights=False)
        x = self.base_block.dropout(x)  # existing pre-residual dropout
        x = input_tensor + x  # first residual add
        x = self.attn_dropout(x)  # ← after first residual

        y = self.base_block.ln_2(x)
        y = self.base_block.mlp(y)
        x = x + y  # second residual add
        x = self.mlp_dropout(x)  # ← after second residual
        return x


def configure_post_residual_dropouts(model: nn.Module, p: float):
    """
    Wraps each ViT-B/16 EncoderBlock with PostResidualEncoderBlock at dropout
    rate p. If blocks are already wrapped, unwraps first so this is safe to
    call repeatedly during a dropout sweep.
    """
    vit_core = model[1] if isinstance(model, nn.Sequential) else model
    layers = vit_core.encoder.layers

    for i, block in enumerate(layers):
        base = (
            block.base_block if isinstance(block, PostResidualEncoderBlock) else block
        )
        layers[i] = PostResidualEncoderBlock(base, p)


def remove_post_residual_dropouts(model: nn.Module):
    """Unwraps all PostResidualEncoderBlock wrappers, restoring original blocks."""
    vit_core = model[1] if isinstance(model, nn.Sequential) else model
    layers = vit_core.encoder.layers

    for i, block in enumerate(layers):
        if isinstance(block, PostResidualEncoderBlock):
            layers[i] = block.base_block


def configure_dropouts(model: nn.Module, p: float, mode):
    if mode == "existing":
        configure_existing_dropouts(model, p)
    else:
        configure_post_residual_dropouts(model, p)


@torch.inference_mode()
def build_primitive_cache(model: nn.Module, loader: DataLoader) -> list:
    """
    Pre-computes baseline (no-dropout) probabilities and argmax labels.
    Called once per model/dataset pair before the dropout sweep.

    Storing the cache avoids recomputing the no-dropout forward pass
    k*len(DROPOUT_RATES) times.
    """
    model.eval()
    configure_dropouts(
        model, p=0.0, mode=DROPOUT_TYPE
    )  # ensure dropout is off for cache
    cache = []
    for imgs, _ in loader:
        imgs = imgs.to(device)
        probs = F.softmax(model(imgs), dim=1)
        cache.append(
            {
                "probs": probs.cpu(),
                "labels": probs.argmax(dim=1).cpu(),
            }
        )
    return cache


@torch.inference_mode()
def compute_psu_and_shift(
    model: nn.Module,
    loader: DataLoader,
    primitive_cache: list,
    k: int = FORWARD_PASSES,
    seed: int = SEED,
) -> tuple:
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()

    all_scores = []
    shift_count = 0
    total_pairs = 0
    seed_everything(seed)
    for (imgs, _), cache_row in zip(loader, primitive_cache):
        imgs = imgs.to(device)
        probs_off = cache_row["probs"].to(device)  # (B, C) — no dropout
        labels_off = cache_row["labels"].to(device)  # (B,)   — argmax class c

        # k stochastic passes with dropout on
        perturbed = []
        for _ in range(k):
            p_on = F.softmax(model(imgs), dim=1)  # (B, C)
            shift_count += (p_on.argmax(dim=1) != labels_off).sum().item()
            perturbed.append(p_on)
        total_pairs += imgs.size(0) * k

        mean_p_on = torch.stack(perturbed, dim=0).mean(dim=0)  # (B, C)

        # PSU: difference in confidence for argmax class c
        diff = probs_off - mean_p_on  # (B, C)
        scores = diff.gather(1, labels_off.view(-1, 1)).squeeze(1)  # (B,)
        all_scores.append(scores.cpu())

    scores = torch.cat(all_scores) if all_scores else torch.empty(0)
    shift_ratio = shift_count / max(total_pairs, 1)
    return scores, shift_ratio


def compute_threshold(val_scores: torch.Tensor, quantile: float = 0.25) -> float:
    return float(torch.quantile(val_scores.to(torch.float32), quantile).item())


def compute_metrics(
    cl_scores: torch.Tensor,
    bd_scores: torch.Tensor,
    threshold: float,
) -> dict:
    """
    Threshold-based metrics. Flagged as backdoor if PSU < threshold.
    TPR = fraction of backdoor samples correctly flagged.
    FPR = fraction of clean samples incorrectly flagged.
    """
    tp = (bd_scores < threshold).float().mean().item()
    fp = (cl_scores < threshold).float().mean().item()
    return {"tpr": float(tp), "fpr": float(fp), "threshold": float(threshold)}


def get_clean_val_subset(cl_test, bd_test, val_size=2000, seed=0):
    original_labels = np.array([cl_test[i][1] for i in range(len(cl_test))])
    indices = np.arange(len(cl_test))

    val_idx, eval_idx = train_test_split(
        indices,
        test_size=(len(indices) - val_size),
        stratify=original_labels,
        random_state=seed,
    )

    val_clean = Subset(cl_test, val_idx.tolist())
    cl_eval = Subset(cl_test, eval_idx.tolist())
    bd_eval = Subset(bd_test, eval_idx.tolist())

    return val_clean, cl_eval, bd_eval


def enable_dropout(model):
    for module in model.modules():
        if isinstance(module, (nn.Dropout, nn.Dropout2d, nn.Dropout3d)):
            module.train()


@torch.inference_mode()
def compute_clean_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    dtype = next(model.parameters()).dtype
    device = next(model.parameters()).device
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        imgs, labels = imgs.to(dtype), labels.to(torch.long)
        correct += (model(imgs).argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return correct / total if total > 0 else 0.0

## Run all experiments

In [ ]:
def evaluate_psbd_performance(
    cl_scores: torch.Tensor,
    bd_scores: torch.Tensor,
    threshold: float,
) -> tuple:
    all_scores = torch.cat([-cl_scores, -bd_scores]).numpy()
    labels = np.concatenate([np.zeros(len(cl_scores)), np.ones(len(bd_scores))])
    auroc = roc_auc_score(labels, all_scores)

    tpr = (bd_scores < threshold).float().mean().item()
    fpr = (cl_scores < threshold).float().mean().item()
    return float(auroc), float(tpr), float(fpr)


def scores_path(exp_dir: str, p: float) -> str:
    s = f"scores_{p}".replace(".", "_") + ".npz"
    return os.path.join(exp_dir, s)


def save_scores(exp_dir: str, p: float, val_scores, cl_scores, bd_scores):
    np.savez_compressed(
        scores_path(exp_dir, p),
        val=val_scores.float().numpy(),
        clean=cl_scores.float().numpy(),
        backdoor=bd_scores.float().numpy(),
    )


def load_scores(exp_dir: str, p: float) -> tuple:
    path = scores_path(exp_dir, p)
    d = np.load(path)
    return (
        torch.tensor(d["val"]),
        torch.tensor(d["clean"]),
        torch.tensor(d["backdoor"]),
    )


def save_metrics_row(exp_dir: str, row: dict):
    path = os.path.join(exp_dir, "metrics.json")
    rows = []
    if os.path.exists(path):
        with open(path) as f:
            rows = json.load(f)
    rows.append(row)
    with open(path, "w") as f:
        json.dump(rows, f, indent=2)


def sweep_attack(
    folder_name: str,
    val_loader: DataLoader,
    cl_loader: DataLoader,
    bd_loader: DataLoader,
    device: torch.device,
    dropout_mode: str = "existing",
    dropout_rates: list = DROPOUT_RATES,
    quantiles: list = PSBD_QUANTILES,
    results_dir: str = RESULTS_DIR,
):
    exp_dir = os.path.join(results_dir, folder_name)
    os.makedirs(exp_dir, exist_ok=True)
    vit_model = load(f"vit_b_16_weights/{folder_name}/attack_result.pt")

    # Model sanity check
    configure_existing_dropouts(vit_model, p=0.0)  # ensure eval mode baseline
    asr = compute_asr(vit_model, bd_loader)
    ca = compute_clean_accuracy(vit_model, cl_loader)
    print(f"\n{'=' * 55}")
    print(f"{folder_name}: ASR={asr:.4f}, CA={ca:.4f}, mode={dropout_mode}")

    # Build caches once (no dropout)
    val_cache = build_primitive_cache(vit_model, val_loader)
    cl_cache = build_primitive_cache(vit_model, cl_loader)
    bd_cache = build_primitive_cache(vit_model, bd_loader)

    handles = []

    for p in dropout_rates:
        configure_dropouts(vit_model, p, dropout_mode)
        val_scores, val_shift = compute_psu_and_shift(vit_model, val_loader, val_cache)
        cl_scores, cl_shift = compute_psu_and_shift(vit_model, cl_loader, cl_cache)
        bd_scores, bd_shift = compute_psu_and_shift(vit_model, bd_loader, bd_cache)
        val_scores, cl_scores, bd_scores = (
            val_scores.float(),
            cl_scores.float(),
            bd_scores.float(),
        )

        save_scores(exp_dir, p, val_scores, cl_scores, bd_scores)

        print(
            f"\tp={p:.2f}, σ_clean={cl_shift:.3f}, σ_bd={bd_shift:.3f}"
            f", gap={cl_shift - bd_shift:.3f}"
        )

        for q in quantiles:
            threshold = float(torch.quantile(val_scores, q).item())
            auroc, tpr, fpr = evaluate_psbd_performance(cl_scores, bd_scores, threshold)
            asr_after = (1 - tpr) * asr

            row = {
                "folder_name": folder_name,
                "dropout_mode": dropout_mode,
                "dropout_rate": p,
                "quantile": q,
                "auroc": round(auroc, 4),
                "tpr": round(tpr, 4),
                "fpr": round(fpr, 4),
                "threshold": round(threshold, 6),
                "asr_before": round(asr, 4),
                "asr_after": round(asr_after, 4),
                "clean_accuracy": round(ca, 4),
                "shift_clean_val": round(val_shift, 4),
                "shift_clean": round(cl_shift, 4),
                "shift_backdoor": round(bd_shift, 4),
            }
            save_metrics_row(exp_dir, row)

            if q == 0.25:  # print only default quantile to keep output readable
                print(f"\tq={q}, AUROC={auroc:.3f}, TPR={tpr:.3f}, FPR={fpr:.3f}")

    # Restore model to clean state after sweep
    if dropout_mode == "post_residual":
        remove_post_residual_dropouts(vit_model)
    configure_existing_dropouts(vit_model, p=0.0)


def run_all_attacks(
    dropout_mode: str = "existing",
    attack_folders: list = ATTACK_FOLDERS,
    results_dir: str = RESULTS_DIR,
):
    os.makedirs(results_dir, exist_ok=True)
    failed = []

    for folder_name in attack_folders:
        checkpoint = f"vit_b_16_weights/{folder_name}/attack_result.pt"
        if not os.path.exists(checkpoint):
            print(f"Skipping {folder_name} — checkpoint not found")
            continue

        try:
            parts = folder_name.split("_")
            dataset = parts[0]  # cifar10 / cifar100 / gtsrb
            transform = get_transform(dataset)

            train_ds, test_ds = load_clean_datasets(dataset, transform)
            bd_test_ds, cl_test_ds = get_backdoor_splits(
                folder_name, test_ds, transform
            )

            val_clean, cl_eval, bd_eval = get_clean_val_subset(
                cl_test_ds, bd_test_ds, val_size=CLEAN_VAL_SIZE, seed=SEED
            )
            cl_eval_balanced, bd_eval_balanced = get_balanced_subset(
                cl_eval, bd_eval, EXAMPLES_PER_CLASS, SEED
            )

            val_loader = DataLoader(val_clean, batch_size=BATCH_SIZE, shuffle=False)
            cl_loader = DataLoader(
                cl_eval_balanced, batch_size=BATCH_SIZE, shuffle=False
            )
            bd_loader = DataLoader(
                bd_eval_balanced, batch_size=BATCH_SIZE, shuffle=False
            )

            sweep_attack(
                folder_name,
                val_loader,
                cl_loader,
                bd_loader,
                device,
                dropout_mode=dropout_mode,
                dropout_rates=DROPOUT_RATES,
                quantiles=PSBD_QUANTILES,
                results_dir=results_dir,
            )

        except Exception as e:
            print(f"FAILED: {folder_name}: {e}")
            failed.append({"folder_name": folder_name, "error": str(e)})

    if failed:
        print(f"\nFailed: {[x['folder_name'] for x in failed]}")

    return failed

In [20]:
SEED = 0
TRIGGER_LABEL = 0
BATCH_SIZE = 16
CLEAN_VAL_SIZE = 2000
EXAMPLES_PER_CLASS = 150
FORWARD_PASSES = 5
PSBD_QUANTILES = [0.05, 0.1, 0.15, 0.20, 0.25]
# DROPOUT_RATES = [i / 10.0 for i in range(1, 10)]
DROPOUT_RATES = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
VIT_WEIGHTS_DIR = "vit_b_16_weights"
RAW_DATA_DIR = "raw_data"
# RESULTS_DIR = "/content/drive/MyDrive/PSBD/experiments"
DROPOUT_TYPE = "existing"
RESULTS_DIR = "experiments/" + DROPOUT_TYPE
os.makedirs(RESULTS_DIR, exist_ok=True)
ATTACK_FOLDERS = [
    "cifar10_blind_0_1",
    "cifar10_bpp_0_1",
    "cifar10_inputaware_0_1",
    "cifar10_lc_0_1",
    "cifar10_lf_0_1",
    "cifar10_bpp_0_05",
    "cifar10_inputaware_0_05",
    "cifar10_lc_0_05",
    "cifar10_lf_0_05",
    "cifar10_bpp_0_01",
    "cifar10_inputaware_0_01",
    "cifar10_lc_0_01",
    "cifar10_lf_0_01",
]

run_all_attacks(
    dropout_mode=DROPOUT_TYPE,
    attack_folders=ATTACK_FOLDERS,
    results_dir=RESULTS_DIR,
)

/home/spiderman/.venvs/research/lib64/python3.14/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/spiderman/.venvs/research/lib64/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


FAILED: cifar10_blind_0_1: No PNG files found in vit_b_16_weights/cifar10_blind_0_1/bd_test_dataset


Seed set to 0



cifar10_bpp_0_1: ASR=0.9978, CA=0.9059, mode=existing


Seed set to 0
Seed set to 0
Seed set to 0
Seed set to 0


...

Seed set to 0
Seed set to 0
Seed set to 0


	p=0.40, σ_clean=0.964, σ_bd=0.964, gap=0.000
	q=0.25, AUROC=0.500, TPR=0.239, FPR=0.239


Seed set to 0
Seed set to 0


	p=0.50, σ_clean=0.956, σ_bd=0.956, gap=0.000
	q=0.25, AUROC=0.500, TPR=0.247, FPR=0.247

Failed: ['cifar10_blind_0_1']


[{'folder_name': 'cifar10_blind_0_1',
  'error': 'No PNG files found in vit_b_16_weights/cifar10_blind_0_1/bd_test_dataset'}]

In [ ]:
ATTACK_FOLDERS = [
    "cifar100_blind_0_1",
    "cifar100_bpp_0_1",
    "cifar100_inputaware_0_1",
    "cifar100_lc_0_1",
    "cifar100_lf_0_1",
    "cifar100_bpp_0_05",
    "cifar100_inputaware_0_05",
    "cifar100_lc_0_05",
    "cifar100_lf_0_05",
    "cifar100_bpp_0_01",
    "cifar100_inputaware_0_01",
    "cifar100_lc_0_01",
    "cifar100_lf_0_01",
]

run_all_attacks(
    dropout_mode=DROPOUT_TYPE,
    attack_folders=ATTACK_FOLDERS,
    results_dir=RESULTS_DIR,
)

/home/spiderman/.venvs/research/lib64/python3.14/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
/home/spiderman/.venvs/research/lib64/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


FAILED: cifar100_blind_0_1: No PNG files found in vit_b_16_weights/cifar100_blind_0_1/bd_test_dataset


Seed set to 0



cifar100_bpp_0_1: ASR=0.9959, CA=0.7371, mode=existing


Seed set to 0
Seed set to 0
Seed set to 0
Seed set to 0


...

Seed set to 0
Seed set to 0
Seed set to 0


	p=0.20, σ_clean=0.833, σ_bd=0.076, gap=0.757
	q=0.25, AUROC=0.908, TPR=0.884, FPR=0.252


Seed set to 0
Seed set to 0
Seed set to 0


	p=0.30, σ_clean=0.974, σ_bd=0.135, gap=0.839
	q=0.25, AUROC=0.805, TPR=0.726, FPR=0.255


Seed set to 0
Seed set to 0


In [21]:
ATTACK_FOLDERS = [
    "gtsrb_blind_0_1",
    "gtsrb_bpp_0_01",
    "gtsrb_bpp_0_05",
    "gtsrb_bpp_0_1",
    "gtsrb_inputaware_0_01",
    "gtsrb_inputaware_0_05",
    "gtsrb_inputaware_0_1",
    "gtsrb_lf_0_01",
    "gtsrb_lf_0_05",
    "gtsrb_lf_0_1",
    "gtsrb_lira_0_1",
    "tiny_lf_0_01",
    "tiny_lf_0_05",
    "tiny_lf_0_1",
    "tiny_lira_0_1",
]

run_all_attacks(
    dropout_mode=DROPOUT_TYPE,
    attack_folders=ATTACK_FOLDERS,
    results_dir=RESULTS_DIR,
)

/home/spiderman/.venvs/research/lib64/python3.14/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


FAILED: gtsrb_blind_0_1: No PNG files found in vit_b_16_weights/gtsrb_blind_0_1/bd_test_dataset


Seed set to 0



gtsrb_bpp_0_01: ASR=0.0397, CA=0.8539, mode=existing


Seed set to 0
Seed set to 0
Seed set to 0
Seed set to 0


...

Seed set to 0
Seed set to 0
Seed set to 0


	p=0.40, σ_clean=0.967, σ_bd=0.967, gap=0.000
	q=0.25, AUROC=0.500, TPR=0.225, FPR=0.225


Seed set to 0
Seed set to 0


	p=0.50, σ_clean=0.968, σ_bd=0.968, gap=0.000
	q=0.25, AUROC=0.500, TPR=0.227, FPR=0.227
Skipping tiny_lira_0_1 — checkpoint not found

Failed: ['gtsrb_blind_0_1']


[{'folder_name': 'gtsrb_blind_0_1',
  'error': 'No PNG files found in vit_b_16_weights/gtsrb_blind_0_1/bd_test_dataset'}]